# `steam_indie_review_histogram.csv` 전처리

## 전처리 방향

| 구분 | 처리 내용 |
|---|---|
| 기본 전처리 | 문자열 컬럼 앞뒤 공백 정리 |
| 기본 전처리 | `release_date`, `hist_start_date`, `hist_end_date`, `date` 날짜형 변환 |
| 기본 전처리 | `appid`, `recommendations_up`, `recommendations_down` 숫자형 타입 통일 |
| 기본 전처리 | 결측치 및 이상치 확인 |
| 파생 컬럼 생성 | `review_date` 생성 |
| 파생 컬럼 생성 | `recommendations_total = recommendations_up + recommendations_down` 생성 |
| 파생 컬럼 생성 | 출시일 기준 경과일 `days_from_release` 생성 |
| 파생 컬럼 생성 | 출시일 기준 구간 `release_period` 생성 |
| 파생 컬럼 생성 | `appid + review_date + data_type` 기준 `histogram_key` 생성 |

## 전처리 저장
| 저장 단계 | 저장 파일 | 포함 내용 |
|---|---|---|
| 1차 저장 | `steam_indie_review_histogram_cleaned_lee_v2.csv` | 문자열 공백 정리, 날짜형 변환, 숫자형 타입 통일, 결측치/이상치 확인까지 반영한 기본 전처리 데이터 |
| 2차 저장 | `steam_indie_review_histogram_preprocessed_lee_v2.csv` | 1차 저장 데이터에 분석용 파생 컬럼을 추가한 최종 전처리 데이터 |

2차 저장은 지워도 됩니다.

In [26]:
# 1. 라이브러리 호출
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

# 보기 옵션
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

# 2. 파일 경로 설정 및 데이터 읽기

In [27]:
# 0. 경로 설정

# 프로젝트 루트 직접 지정
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정합니다.
ROOT = Path(r"C:\Users\joon5\Documents\github\steam-indie-game-analysis")

# 원천/소스 파일이 들어있는 폴더
DATA_RAW_DIR = ROOT / "data" / "raw"

# 전처리 저장 폴더
DATA_DIR = ROOT / "data" / "processed"

# 전처리 결과 저장 경로
OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 저장 파일명
# 1차 저장: 컬럼 생성 전, 기본 전처리만 완료한 데이터
CLEANED_PATH = OUTPUT_DIR / "steam_indie_review_histogram_cleaned_lee_v2.csv"

# 2차 저장: 분석용 파생 컬럼까지 생성한 최종 데이터
PREPROCESSED_PATH = OUTPUT_DIR / "steam_indie_review_histogram_preprocessed_lee_v2.csv"

print("호출 폴더")
print("DATA_RAW_DIR:", DATA_RAW_DIR)

print("\n저장 폴더")
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CLEANED_PATH:", CLEANED_PATH)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)


# GAMES_RAW_PATH = DATA_RAW_DIR / "steam_indie_games.csv"
# REVIEWS_RAW_PATH = DATA_RAW_DIR / "steam_indie_reviews.csv"
# REVIEW_SUMMARY_RAW_PATH = DATA_RAW_DIR / "steam_indie_review_summary.csv"
REVIEW_HISTOGRAM_RAW_PATH = DATA_RAW_DIR / "steam_indie_review_histogram.csv"
# TAGS_RAW_PATH = DATA_RAW_DIR / "steam_indie_tags.csv"


호출 폴더
DATA_RAW_DIR: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\raw

저장 폴더
OUTPUT_DIR: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed
CLEANED_PATH: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_review_histogram_cleaned_lee_v2.csv
PREPROCESSED_PATH: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_review_histogram_preprocessed_lee_v2.csv


In [28]:
# 원본 데이터 읽기
# df_raw는 원본 보존용, 실제 전처리는 df_hist에서 진행
# df_games_raw = pd.read_csv(GAMES_RAW_PATH)                          # 게임 메타데이터 원본
# df_reviews_raw = pd.read_csv(REVIEWS_RAW_PATH)                      # 리뷰 본문 원본
# df_summary_raw = pd.read_csv(REVIEW_SUMMARY_RAW_PATH)               # 리뷰 요약 원본
df_hist_raw = pd.read_csv(REVIEW_HISTOGRAM_RAW_PATH)                  # 리뷰 히스토그램 원본
# df_tags_raw = pd.read_csv(TAGS_RAW_PATH)                            # 태그 원본

# 이후 전처리에서 사용할 복사본
# df_games = df_games_raw.copy()
# df_reviews = df_reviews_raw.copy()
# df_summary = df_summary_raw.copy()
df_hist = df_hist_raw.copy()
# df_tags = df_tags_raw.copy()

# 데이터 크기 확인
# print("df_games:", df_games.shape)
# display(df_games.head())
# print("df_reviews:", df_reviews.shape)
# display(df_reviews.head())
# print("df_review_summary:", df_summary.shape)
# display(df_review_summary.head())
print("df_review_histogram:", df_hist.shape)
display(df_hist.head())
# print("df_tags:", df_tags.shape)
# display(df_tags.head())

df_review_histogram: (11782, 10)


,appid,name,stratum,release_date,hist_start_date,hist_end_date,date,recommendations_up,recommendations_down,data_type
0,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-07,0,0,recent
1,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-08,0,0,recent
2,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-09,0,0,recent
3,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-10,0,0,recent
4,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-11,0,0,recent


# 3. 전처리 전 기본 확인

In [29]:
def check_basic_info(df, df_name, exclude_cols=None):
    """행/열 수, 완전 중복 행, 컬럼별 타입/결측/고유값을 한 번에 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 기본 정보 / 타입 / 결측치 확인")
    print(f"{'='*80}\n")

    # 제외할 컬럼 반영
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # dict, list, set 같은 해시 불가능 값이 들어있는 컬럼은 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)

    # 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })

    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df_copied) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df_copied) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    }).sort_values(by=['결측치 개수', '고유값 개수'], ascending=[False, False])

    print("[전체 요약]")
    display(overview_df)

    print("[컬럼별 요약]")
    display(summary_df)

    print("[상위 5행]")
    display(df_copied.head())

In [30]:
def check_id_duplicates(df, col_name, df_name, top_n=10):
    """
    기준 키로 쓸 컬럼의 중복 여부를 확인한다.

    col_name에 문자열 1개를 넣으면 단일 키를 확인하고,
    리스트를 넣으면 여러 컬럼 조합 기준으로 중복을 확인한다.

    예)
    check_id_duplicates(df, "appid", "df_hist")
    check_id_duplicates(df, ["appid", "review_date", "data_type"], "df_hist")
    """
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 값 중복 확인")
    print(f"{'='*80}")

    df_copied = df.copy()

    # 단일 컬럼인 경우
    if isinstance(col_name, str):
        if col_name not in df_copied.columns:
            print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
            return

        key_series = df_copied[col_name]
        key_label = col_name

    # 여러 컬럼 조합인 경우
    else:
        missing_cols = [col for col in col_name if col not in df_copied.columns]

        if missing_cols:
            print(f"존재하지 않는 컬럼이 있습니다: {missing_cols}")
            return

        key_series = df_copied[col_name].astype(str).agg(" | ".join, axis=1)
        key_label = " + ".join(col_name)

    duplicate_count = key_series.duplicated().sum()

    print("전체 행 수:", len(df_copied))
    print(f"{key_label} 고유 개수:", key_series.nunique(dropna=True))
    print(f"중복 {key_label} 개수:", duplicate_count)

    if duplicate_count > 0:
        print()
        print("[중복 상위 값]")
        dup_summary = key_series.value_counts(dropna=False).reset_index()
        dup_summary.columns = [key_label, "등장 횟수"]
        display(dup_summary[dup_summary["등장 횟수"] > 1].head(top_n))
    else:
        print("중복 값이 없습니다.")

# 이 코드를 진짜 몇번이고 개량하는지 모르겠네

In [31]:
check_basic_info(df_hist, "df_hist")


df_hist의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,11782
1,열 개수,10
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
date,str,11782,100.0,0,0.0,995
recommendations_up,int64,11782,100.0,0,0.0,321
appid,int64,11782,100.0,0,0.0,200
name,str,11782,100.0,0,0.0,200
release_date,str,11782,100.0,0,0.0,177
hist_start_date,str,11782,100.0,0,0.0,177
hist_end_date,str,11782,100.0,0,0.0,112
recommendations_down,int64,11782,100.0,0,0.0,87
stratum,str,11782,100.0,0,0.0,24
data_type,str,11782,100.0,0,0.0,2


[상위 5행]


,appid,name,stratum,release_date,hist_start_date,hist_end_date,date,recommendations_up,recommendations_down,data_type
0,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-07,0,0,recent
1,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-08,0,0,recent
2,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-09,0,0,recent
3,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-10,0,0,recent
4,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-11,0,0,recent


In [32]:
check_id_duplicates(df_hist, "appid", "df_hist")


df_hist의 appid 값 중복 확인
전체 행 수: 11782
appid 고유 개수: 200
중복 appid 개수: 11582

[중복 상위 값]


,appid,등장 횟수
0,402160,143
1,571740,141
2,1996010,133
3,2629330,132
4,2703850,132
5,444690,127
6,1640630,126
7,2393770,125
8,743130,118
9,2581050,115


# 4. 전처리

## 4-1. 문자열 컬럼 공백 정리

In [33]:
# 문자열 컬럼만 선택해서 앞뒤 공백을 제거한다.
# 원본 의미를 크게 바꾸지 않기 위해 기본적인 strip 처리만 수행한다.
string_cols = df_hist.select_dtypes(include=["object"]).columns.tolist()

print("문자열 공백 정리 대상 컬럼")
print(string_cols)

문자열 공백 정리 대상 컬럼
['name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'data_type']


C:\Users\joon5\AppData\Local\Temp\ipykernel_18964\1578558134.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df_hist.select_dtypes(include=["object"]).columns.tolist()


In [34]:
# 공백 정리 전 상태 확인
space_before_rows = []

for col in string_cols:
    s = df_hist[col].dropna().astype(str)

    space_before_rows.append({
        "column": col,
        "blank_count_before": s.str.strip().eq("").sum(),
        "leading_trailing_space_before": s.ne(s.str.strip()).sum()
    })

space_before_df = pd.DataFrame(space_before_rows)
display(space_before_df)

,column,blank_count_before,leading_trailing_space_before
0,name,0,0
1,stratum,0,0
2,release_date,0,0
3,hist_start_date,0,0
4,hist_end_date,0,0
5,date,0,0
6,data_type,0,0


In [35]:
# 문자열 컬럼 앞뒤 공백 제거
for col in string_cols:
    df_hist[col] = df_hist[col].astype("string").str.strip()

# 게임명은 연속 공백만 하나의 공백으로 정리
if "name" in df_hist.columns:
    df_hist["name"] = df_hist["name"].str.replace(r"\s+", " ", regex=True)

# data_type은 rollups/recent처럼 값 비교에 사용되므로 소문자로 통일
if "data_type" in df_hist.columns:
    df_hist["data_type"] = df_hist["data_type"].str.lower()

In [36]:
# 공백 정리 후 상태 확인
space_after_rows = []

for col in string_cols:
    s = df_hist[col].dropna().astype(str)

    space_after_rows.append({
        "column": col,
        "blank_count_after": s.str.strip().eq("").sum(),
        "leading_trailing_space_after": s.ne(s.str.strip()).sum()
    })

space_after_df = pd.DataFrame(space_after_rows)
display(space_after_df)

,column,blank_count_after,leading_trailing_space_after
0,name,0,0
1,stratum,0,0
2,release_date,0,0
3,hist_start_date,0,0
4,hist_end_date,0,0
5,date,0,0
6,data_type,0,0


review_score_desc 컬럼이 없어 이 단계는 건너뜁니다.


## 4-3. 날짜형 컬럼 변환

In [38]:
# 날짜 컬럼을 datetime으로 변환한다.
# errors='coerce'를 사용하면 변환이 불가능한 값은 NaT로 처리된다.
date_cols = ["release_date", "hist_start_date", "hist_end_date", "date"]

for col in date_cols:
    if col in df_hist.columns:
        df_hist[col] = pd.to_datetime(df_hist[col], errors="coerce")

# 날짜 변환 결과 확인
date_check_rows = []

for col in date_cols:
    if col in df_hist.columns:
        date_check_rows.append({
            "column": col,
            "dtype": str(df_hist[col].dtype),
            "missing_count": df_hist[col].isna().sum(),
            "min_date": df_hist[col].min(),
            "max_date": df_hist[col].max()
        })

date_check_df = pd.DataFrame(date_check_rows)
display(date_check_df)

,column,dtype,missing_count,min_date,max_date
0,release_date,datetime64[us],0,2023-01-06,2025-11-08
1,hist_start_date,datetime64[us],0,2015-09-17,2025-06-04
2,hist_end_date,datetime64[us],0,2023-09-24,2026-04-29
3,date,datetime64[us],0,2015-09-01,2026-04-29


# 4-4 숫자형 컬럼 타입 통일

In [39]:
# 리뷰 수 컬럼을 숫자형으로 변환한다.
numeric_cols = ["appid", "recommendations_up", "recommendations_down"]

for col in numeric_cols:
    if col in df_hist.columns:
        df_hist[col] = pd.to_numeric(df_hist[col], errors="coerce")

# 결측이 없다는 점검 결과를 바탕으로 정수형으로 변환한다.
df_hist["appid"] = df_hist["appid"].astype("int64")
df_hist["recommendations_up"] = df_hist["recommendations_up"].astype("int64")
df_hist["recommendations_down"] = df_hist["recommendations_down"].astype("int64")

display(
    df_hist[[
            "appid", 
            "name", 
            "date", 
            "data_type",
            "recommendations_up", 
            "recommendations_down"]
    ].head()
)

,appid,name,date,data_type,recommendations_up,recommendations_down
0,402160,Star Command Galaxies,2026-03-07,recent,0,0
1,402160,Star Command Galaxies,2026-03-08,recent,0,0
2,402160,Star Command Galaxies,2026-03-09,recent,0,0
3,402160,Star Command Galaxies,2026-03-10,recent,0,0
4,402160,Star Command Galaxies,2026-03-11,recent,0,0


## 4-5. 결측치 및 이상치 후보 확인

In [40]:
# 결측치와 이상치 후보를 확인한다.
# review_histogram 데이터에서는 리뷰 수가 날짜별 집계값이므로,
# 큰 값 자체는 흥행 규모를 반영할 수 있어 바로 이상치로 제거하지 않는다.

missing_check_df = (
    df_hist
    .isna()
    .sum()
    .reset_index()
)

missing_check_df.columns = ["column", "missing_count"]
missing_check_df["missing_ratio"] = missing_check_df["missing_count"] / len(df_hist)

display(missing_check_df)

# 명확한 오류 가능성이 있는 값만 이상치 후보로 확인한다.
outlier_check_rows = []

if "recommendations_up" in df_hist.columns:
    outlier_check_rows.append({
        "check_item": "recommendations_up < 0",
        "row_count": (df_hist["recommendations_up"] < 0).sum()
    })

if "recommendations_down" in df_hist.columns:
    outlier_check_rows.append({
        "check_item": "recommendations_down < 0",
        "row_count": (df_hist["recommendations_down"] < 0).sum()
    })

if {"hist_start_date", "hist_end_date"}.issubset(df_hist.columns):
    outlier_check_rows.append({
        "check_item": "hist_start_date > hist_end_date",
        "row_count": (df_hist["hist_start_date"] > df_hist["hist_end_date"]).sum()
    })

if {"date", "hist_start_date", "hist_end_date"}.issubset(df_hist.columns):
    outlier_check_rows.append({
        "check_item": "date가 hist_start_date ~ hist_end_date 범위 밖(rollups 월 기준 가능)",
        "row_count": ((df_hist["date"] < df_hist["hist_start_date"]) | (df_hist["date"] > df_hist["hist_end_date"])).sum()
    })

outlier_check_df = pd.DataFrame(outlier_check_rows)
display(outlier_check_df)

,column,missing_count,missing_ratio
0,appid,0,0.0
1,name,0,0.0
2,stratum,0,0.0
3,release_date,0,0.0
4,hist_start_date,0,0.0
5,hist_end_date,0,0.0
6,date,0,0.0
7,recommendations_up,0,0.0
8,recommendations_down,0,0.0
9,data_type,0,0.0


,check_item,row_count
0,recommendations_up < 0,0
1,recommendations_down < 0,0
2,hist_start_date > hist_end_date,0
3,date가 hist_start_date ~ hist_end_date 범위 밖(rol...,89


## 4-6. 1차 저장: 컬럼 생성 전 기본 전처리 데이터

In [41]:
# 1차 CSV 저장
# 이 파일은 분석용 파생 컬럼을 만들기 전의 기본 전처리 결과다.
df_hist.to_csv(CLEANED_PATH, index=False, encoding="utf-8-sig")

print("1차 저장 완료:", CLEANED_PATH)
print("1차 저장 shape:", df_hist.shape)

print()
print("[1차 저장 컬럼 목록]")
print(df_hist.columns.tolist())

1차 저장 완료: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_review_histogram_cleaned_lee_v2.csv
1차 저장 shape: (11782, 10)

[1차 저장 컬럼 목록]
['appid', 'name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type']


# 5. 파생 컬럼 생성
여기서 부턴 파생컬럼 생성

## 5-1. 리뷰 기준 날짜 컬럼 생성

In [42]:
# date는 리뷰가 집계된 날짜이므로 분석용 이름을 하나 더 만든다.
df_hist["review_date"] = df_hist["date"]

display(
    df_hist[["appid", "name", "date", "review_date"]].head()
)

,appid,name,date,review_date
0,402160,Star Command Galaxies,2026-03-07,2026-03-07
1,402160,Star Command Galaxies,2026-03-08,2026-03-08
2,402160,Star Command Galaxies,2026-03-09,2026-03-09
3,402160,Star Command Galaxies,2026-03-10,2026-03-10
4,402160,Star Command Galaxies,2026-03-11,2026-03-11


## 5-2. 리뷰 수 합계 컬럼 생성

In [43]:
# 긍정 리뷰 수 + 부정 리뷰 수
df_hist["recommendations_total"] = (
    df_hist["recommendations_up"] + df_hist["recommendations_down"]
)

display(
    df_hist[[
            "appid", 
            "name", 
            "review_date", 
            "data_type",
            "recommendations_up", 
            "recommendations_down", 
            "recommendations_total"]
    ].head()
)

,appid,name,review_date,data_type,recommendations_up,recommendations_down,recommendations_total
0,402160,Star Command Galaxies,2026-03-07,recent,0,0,0
1,402160,Star Command Galaxies,2026-03-08,recent,0,0,0
2,402160,Star Command Galaxies,2026-03-09,recent,0,0,0
3,402160,Star Command Galaxies,2026-03-10,recent,0,0,0
4,402160,Star Command Galaxies,2026-03-11,recent,0,0,0


## 5-3. 출시일 기준 경과일 생성

In [44]:
df_hist["days_from_release"] = (
    df_hist["review_date"] - df_hist["release_date"]
).dt.days

# 출시일보다 앞선 리뷰는 바로 제거하지 않고 확인만 한다.
# 얼리액세스, 출시 전 리뷰, 날짜 기준 차이 가능성이 있기 때문이다.
print("days_from_release 결측 수:", df_hist["days_from_release"].isna().sum())
print("출시일보다 이른 리뷰 행 수:", (df_hist["days_from_release"] < 0).sum())

display(
    df_hist[
        ["appid", "name", "release_date", "review_date", "days_from_release"]
    ].sort_values("days_from_release").head(10)
)

days_from_release 결측 수: 0
출시일보다 이른 리뷰 행 수: 990


,appid,name,release_date,review_date,days_from_release
30,402160,Star Command Galaxies,2024-08-28,2015-09-01,-3284
31,402160,Star Command Galaxies,2024-08-28,2015-10-01,-3254
32,402160,Star Command Galaxies,2024-08-28,2015-11-01,-3223
236,444690,TRAPPED,2025-06-16,2016-09-01,-3210
33,402160,Star Command Galaxies,2024-08-28,2015-12-01,-3193
237,444690,TRAPPED,2025-06-16,2016-10-01,-3180
34,402160,Star Command Galaxies,2024-08-28,2016-01-01,-3162
238,444690,TRAPPED,2025-06-16,2016-11-01,-3149
504,597920,Survivalizm - The Animal Simulator,2025-11-08,2017-04-01,-3143
35,402160,Star Command Galaxies,2024-08-28,2016-02-01,-3131


## 5-4. 출시일 기준 구간 컬럼 생성

In [45]:
# 출시일 기준으로 리뷰가 어느 구간에 속하는지 구분한다.
bins = [-np.inf, -1, 7, 30, 90, 180, np.inf]
labels = [
    "before_release",
    "D0_7",
    "D8_30",
    "D31_90",
    "D91_180",
    "D181_plus"
]

df_hist["release_period"] = pd.cut(
    df_hist["days_from_release"],
    bins=bins,
    labels=labels
).astype("string")

release_period_summary = (
    df_hist["release_period"]
    .value_counts(dropna=False)
    .reset_index()
)

release_period_summary.columns = ["release_period", "row_count"]
display(release_period_summary)

,release_period,row_count
0,D181_plus,8737
1,before_release,990
2,D91_180,875
3,D31_90,627
4,D8_30,331
5,D0_7,222


## 5-5. 기준 키 생성 및 중복 확인

In [46]:
# histogram 데이터는 appid만으로는 중복이 정상이다.
# 분석 기준 키는 appid + review_date + data_type 조합으로 만든다.
df_hist["histogram_key"] = (
    df_hist["appid"].astype(str)
    + "_"
    + df_hist["review_date"].dt.strftime("%Y-%m-%d").astype(str)
    + "_"
    + df_hist["data_type"].astype(str)
)

display(
    df_hist[
        ["appid", "review_date", "data_type", "histogram_key"]
    ].head()
)

,appid,review_date,data_type,histogram_key
0,402160,2026-03-07,recent,402160_2026-03-07_recent
1,402160,2026-03-08,recent,402160_2026-03-08_recent
2,402160,2026-03-09,recent,402160_2026-03-09_recent
3,402160,2026-03-10,recent,402160_2026-03-10_recent
4,402160,2026-03-11,recent,402160_2026-03-11_recent


In [47]:
# appid 단독 중복 확인
check_id_duplicates(df_hist, "appid", "df_hist")

# appid + review_date + data_type 조합 기준 중복 확인
# 별도 check_multi_key_duplicates 함수는 만들지 않고 check_id_duplicates 하나로 확인한다.
check_id_duplicates(df_hist, ["appid", "review_date", "data_type"], "df_hist")


df_hist의 appid 값 중복 확인
전체 행 수: 11782
appid 고유 개수: 200
중복 appid 개수: 11582

[중복 상위 값]


,appid,등장 횟수
0,402160,143
1,571740,141
2,1996010,133
3,2629330,132
4,2703850,132
5,444690,127
6,1640630,126
7,2393770,125
8,743130,118
9,2581050,115



df_hist의 ['appid', 'review_date', 'data_type'] 값 중복 확인
전체 행 수: 11782
appid + review_date + data_type 고유 개수: 11782
중복 appid + review_date + data_type 개수: 0
중복 값이 없습니다.


# 6. 전처리 후 최종 확인 및 저장

In [48]:
check_basic_info(df_hist, "steam_indie_review_histogram_preprocessed")


steam_indie_review_histogram_preprocessed의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,11782
1,열 개수,15
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
histogram_key,str,11782,100.0,0,0.0,11782
days_from_release,int64,11782,100.0,0,0.0,1904
date,datetime64[us],11782,100.0,0,0.0,995
review_date,datetime64[us],11782,100.0,0,0.0,995
recommendations_total,int64,11782,100.0,0,0.0,334
recommendations_up,int64,11782,100.0,0,0.0,321
appid,int64,11782,100.0,0,0.0,200
name,string,11782,100.0,0,0.0,200
release_date,datetime64[us],11782,100.0,0,0.0,177
hist_start_date,datetime64[us],11782,100.0,0,0.0,177


[상위 5행]


,appid,name,stratum,release_date,hist_start_date,hist_end_date,date,recommendations_up,recommendations_down,data_type,review_date,recommendations_total,days_from_release,release_period,histogram_key
0,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-07,0,0,recent,2026-03-07,0,556,D181_plus,402160_2026-03-07_recent
1,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-08,0,0,recent,2026-03-08,0,557,D181_plus,402160_2026-03-08_recent
2,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-09,0,0,recent,2026-03-09,0,558,D181_plus,402160_2026-03-09_recent
3,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-10,0,0,recent,2026-03-10,0,559,D181_plus,402160_2026-03-10_recent
4,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-11,0,0,recent,2026-03-11,0,560,D181_plus,402160_2026-03-11_recent


In [49]:
# 전처리 전후 shape 비교
print("전처리 전 shape:", df_hist_raw.shape)
print("1차 저장 shape:", pd.read_csv(CLEANED_PATH).shape if CLEANED_PATH.exists() else "아직 저장되지 않음")
print("최종 전처리 후 shape:", df_hist.shape)

# 최종 컬럼 목록 확인
print()
print("[최종 컬럼 목록]")
print(df_hist.columns.tolist())

display(df_hist.head())

전처리 전 shape: (11782, 10)
1차 저장 shape: (11782, 10)
최종 전처리 후 shape: (11782, 15)

[최종 컬럼 목록]
['appid', 'name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type', 'review_date', 'recommendations_total', 'days_from_release', 'release_period', 'histogram_key']


,appid,name,stratum,release_date,hist_start_date,hist_end_date,date,recommendations_up,recommendations_down,data_type,review_date,recommendations_total,days_from_release,release_period,histogram_key
0,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-07,0,0,recent,2026-03-07,0,556,D181_plus,402160_2026-03-07_recent
1,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-08,0,0,recent,2026-03-08,0,557,D181_plus,402160_2026-03-08_recent
2,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-09,0,0,recent,2026-03-09,0,558,D181_plus,402160_2026-03-09_recent
3,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-10,0,0,recent,2026-03-10,0,559,D181_plus,402160_2026-03-10_recent
4,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-11,0,0,recent,2026-03-11,0,560,D181_plus,402160_2026-03-11_recent


In [50]:
# 2차 CSV 저장
# 이 파일은 기본 전처리 결과에 분석용 파생 컬럼까지 추가한 최종 전처리 데이터다.
df_hist.to_csv(PREPROCESSED_PATH, index=False, encoding="utf-8-sig")

print("2차 저장 완료:", PREPROCESSED_PATH)
print("2차 저장 shape:", df_hist.shape)

2차 저장 완료: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_review_histogram_preprocessed_lee_v2.csv
2차 저장 shape: (11782, 15)


# 7. 전처리 요약

이번 전처리에서는 `steam_indie_review_histogram.csv`의 구조를 크게 바꾸지 않고,  
분석에 바로 필요한 최소 컬럼만 추가했다.

## 7-1. 1차 저장: 기본 전처리 데이터
`steam_indie_review_histogram_cleaned_lee_v2.csv`는 **컬럼 생성 전 데이터**다.\
원본 컬럼 구조는 유지하되, 분석 전에 필요한 기본 정리만 적용했다.

| 처리 항목 | 결과 |
|---|---|
| 문자열 공백 | 문자열 컬럼 앞뒤 공백 정리 |
| 게임명 공백 | `name` 컬럼의 연속 공백을 하나로 정리 |
| 집계 구분값 | `data_type` 소문자 통일 |
| 날짜 변환 | `release_date`, `hist_start_date`, `hist_end_date`, `date` 날짜형 변환 |
| 숫자형 변환 | `appid`, `recommendations_up`, `recommendations_down` 숫자형 타입 통일 |
| 결측치 확인 | 컬럼별 결측치 개수 및 비율 확인 |
| 이상치 확인 | 음수 리뷰 수, 날짜 범위 오류 등 명확한 오류 후보 확인 |

## 7-2. 2차 저장: 파생 컬럼 포함 최종 데이터

`steam_indie_review_histogram_preprocessed_lee_v2.csv`는 **컬럼 생성 후 데이터**다.\
1차 저장 데이터에 분석에 필요한 파생 컬럼을 추가했다.

| 생성 컬럼 | 결과 |
|---|---|
| `review_date` | 리뷰 집계 기준 날짜 |
| `recommendations_total` | 긍정 리뷰 수 + 부정 리뷰 수 |
| `days_from_release` | 출시일로부터 리뷰 기준 날짜까지 지난 일수 |
| `release_period` | 출시일 기준 경과일 구간 |
| `histogram_key` | `appid + review_date + data_type` 조합 기준 키 |

# 8. 컬럼 설명

## 8-1. 1차 저장 파일 컬럼

`steam_indie_review_histogram_cleaned_lee_v2.csv`

| 컬럼명 | 한 줄 설명 |
|---|---|
| `appid` | Steam 게임 고유 ID |
| `name` | Steam 게임명 |
| `stratum` | 표본 추출 시 사용한 층화 그룹 정보 |
| `release_date` | 게임 출시일 |
| `hist_start_date` | 리뷰 히스토그램 수집 시작일 |
| `hist_end_date` | 리뷰 히스토그램 수집 종료일 |
| `date` | 리뷰 집계 기준 날짜 |
| `recommendations_up` | 해당 날짜 기준 긍정 리뷰 수 |
| `recommendations_down` | 해당 날짜 기준 부정 리뷰 수 |
| `data_type` | 리뷰 집계 방식 구분값 |

## 8-2. 2차 저장 파일에서 추가된 컬럼

`steam_indie_review_histogram_preprocessed_lee_v2.csv`

| 컬럼명 | 한 줄 설명 |
|---|---|
| `review_date` | 분석에 사용할 리뷰 기준 날짜 |
| `recommendations_total` | 긍정 리뷰 수와 부정 리뷰 수를 합친 전체 리뷰 수 |
| `days_from_release` | 출시일로부터 리뷰 기준 날짜까지 지난 일수 |
| `release_period` | 출시일 기준 경과일을 구간화한 값 |
| `histogram_key` | `appid + review_date + data_type`을 합쳐 만든 리뷰 히스토그램 기준 키 |
